In [86]:
import pandas as pd 
import numpy as np 
from glob import glob
import os
import geopandas as gpd

import csv
import json

from ast import literal_eval

from shapely import wkt

# Set option to display all rows (no truncation)
pd.set_option('display.max_rows', None)

# Set option to display all columns (no truncation)
pd.set_option('display.max_columns', None)

In [ ]:
# 1/25 
# - need to add in coincident events (e.g. do the same timestamps have heavy snow and lake effect snow?)
# - note that ncei doesnt have snow squall. Is there a better squall observational dataset? if not this could be a contribution of my work. Ask nick/heather?
# In /home/csutter/DRIVE-clean/weather_events/notebooks/ncei_dataset_analysis.ipynb has the data datetime prep logic, which prepared the data for operational runs, but ALSO it saved out the prepped data for the events in dfs, with proper naming convention to note dates of interest, frequency of pulls (every 15 min), etc. Use these for modeling work! The dfs are: /home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest
# note that we ran the 15 min frequency for 2024 & 2025 for all events
# - regarding that ^ note, that was all set up ( ) and ran for NCEI events.  But did not do this for the NWS warnings dataset. Although those are easy to remember bc i never adjusted the logic, I just did the 5 min intervals, wth 15 min windows on each end of the warning,  every 5 min run, and all years (no filtering). (see /home/csutter/DRIVE-clean/NWS_warnings/notebooks/nws_dataset_analysis.ipynb for the details there). But that was simpler, I only started adjusting the data prep/ model run logic for NCEI data

# - IMPORTANT! If there are no cron logged images at all for a given datetime run instance, nothing will be saved out beyond an empty data_1_images directory. Don't have a clean way to deal with this for tracking this situation yet (since in the ncei_events_ofint directory this event will still be listed, it's just impossible for us to get model runs for it).  For an example of this situation, see /home/csutter/DRIVE-clean/operational_runs/set35_blizzard8_EgNoImgs
# - Also need to keep in mind to remove events that were used to label the data

# NCEI

In [57]:
# Need to read in multiple datasets: 1) NCEI geodataframe with geometries 2) Events csvs (with logic about every 15 min frequency, etc) and 3) camera lat and lons 4) Model run data 
# The first three, 1-3, are easy (in this block of code)
# The fourth will take navigating through operational run directory to find prediction csvs given the dates we pull (more code, below)

#### 1 - NCEI data
d_readin = gpd.read_file("/home/csutter/DRIVE-clean/weather_events/data/ncei_events/ncei_ny_events_clean.gpkg")

d_readin.head(4)

# add timedelta duration col back (using duration_sec col)
# note that you have to do this w/ every gdf
d_readin["duration"] = pd.to_timedelta(d_readin["duration_sec"], unit="s")

# may take ~40 seconds

#### 2 - Identified events
# grab just one file for now, but will need to eventually tie in all of them

events = pd.read_csv("/home/csutter/DRIVE-clean/weather_events/data/ncei_events_ofinterest/blizzard_allyrs_ceilfloor5min_nobuffer_freq5min.csv")
# convert this events df to geopandas df
# Convert the 'geometry' column from strings to actual Shapely objects
events['geometry'] = events['geometry'].apply(wkt.loads)
# Cast as a GeoDataFrame and set the CRS
# (Use the CRS of your original ncei_gdf, usually "EPSG:4269" for NWS data)
events_gdf = gpd.GeoDataFrame(events, geometry='geometry', crs="EPSG:4269")
# Match the points to the NCEI CRS
# This converts the points' coordinates to fit the storm polygons perfectly
events_gdf = events_gdf.to_crs(d_readin.crs)


#### 3 - Cam lat and lons (not sure we need, just load data for now)

cams = pd.read_csv("/home/csutter/DRIVE/site_analysis/_reference/511NY_API_GetCameras_response.csv")

cams = cams[((cams["Disabled"]==False)&(cams["Blocked"]==False))]
print(len(cams))

2371


In [110]:
d_readin.head(4)

,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration
0,14,202202,3,900,202202,4,1600,164922,995747,NEW YORK,36,2022,February,Winter Storm,Z,31,WESTERN CLINTON,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,031,031,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.56300 44.50731, -73.55910 44.479...",1 days 07:00:00
1,15,202202,3,900,202202,4,1600,164922,995749,NEW YORK,36,2022,February,Winter Storm,Z,30,SOUTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 10 to 14 inches wit...,CSV,030,030,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-73.98640 44.70781, -73.96619 44.709...",1 days 07:00:00
2,16,202202,3,900,202202,4,1600,164922,995750,NEW YORK,36,2022,February,Winter Storm,Z,27,NORTHERN FRANKLIN,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 8 to 10 inches with...,CSV,027,027,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-74.66310 44.99891, -74.66100 44.999...",1 days 07:00:00
3,17,202202,3,900,202202,4,1600,164922,995752,NEW YORK,36,2022,February,Winter Storm,Z,29,SOUTHEASTERN ST. LAWRENCE,BTV,2022-02-03 09:00:00,EST-5,2022-02-04 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,None,None,None,None,NaN,NaN,None,None,NaN,None,NaN,None,None,NaN,None,None,NaN,NaN,NaN,NaN,An arctic front draped across northern NY and ...,Total snowfall ranged from 6 to 10 inches with...,CSV,029,029,None,None,2022-02-03 14:00:00,2022-02-04 21:00:00,111600.0,"POLYGON ((-75.06280 44.05041, -75.06920 44.053...",1 days 07:00:00


In [58]:
type(events_gdf)

geopandas.geodataframe.GeoDataFrame

In [56]:
events.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'BEGIN_YEARMONTH', 'BEGIN_DAY',
       'BEGIN_TIME', 'END_YEARMONTH', 'END_DAY', 'END_TIME', 'EPISODE_ID',
       'EVENT_ID', 'STATE', 'STATE_FIPS', 'YEAR', 'MONTH_NAME', 'EVENT_TYPE',
       'CZ_TYPE', 'CZ_FIPS', 'CZ_NAME', 'WFO', 'BEGIN_DATE_TIME',
       'CZ_TIMEZONE', 'END_DATE_TIME', 'INJURIES_DIRECT', 'INJURIES_INDIRECT',
       'DEATHS_DIRECT', 'DEATHS_INDIRECT', 'DAMAGE_PROPERTY', 'DAMAGE_CROPS',
       'SOURCE', 'MAGNITUDE', 'MAGNITUDE_TYPE', 'FLOOD_CAUSE', 'CATEGORY',
       'TOR_F_SCALE', 'TOR_LENGTH', 'TOR_WIDTH', 'TOR_OTHER_WFO',
       'TOR_OTHER_CZ_STATE', 'TOR_OTHER_CZ_FIPS', 'TOR_OTHER_CZ_NAME',
       'BEGIN_RANGE', 'BEGIN_AZIMUTH', 'BEGIN_LOCATION', 'END_RANGE',
       'END_AZIMUTH', 'END_LOCATION', 'BEGIN_LAT', 'BEGIN_LON', 'END_LAT',
       'END_LON', 'EPISODE_NARRATIVE', 'EVENT_NARRATIVE', 'DATA_SOURCE',
       'CZ_FIPS_FORMAT', 'ZONE', 'FIPS', 'FIPS_FORMAT', 'BEGIN_UTC', 'END_UTC',
       'duration_sec', 'geometry', 'duration'

In [53]:
print(type(d_readin))
print(type(events))
print(type(events1))

<class 'geopandas.geodataframe.GeoDataFrame'>
<class 'pandas.core.frame.DataFrame'>
<class 'pandas.core.frame.DataFrame'>


In [3]:
events.head(3)

,Unnamed: 0.1,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
0,118,4768,202201,29,700,202201,29,1330,165058,996866,NEW YORK,36,2022,January,Blizzard,Z,80,SOUTHWEST SUFFOLK,OKX,2022-01-29 07:00:00,EST-5,2022-01-29 13:30:00,0,0,0,0,0.00K,0.00K,Official NWS Observations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"Islip, NY set a record for its highest calenda...",CSV,80,80,NaN,NaN,2022-01-29 12:00:00,2022-01-29 18:30:00,23400.0,MULTIPOLYGON (((-73.42420196499995 40.62370681...,0 days 06:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,"DatetimeIndex(['2022-01-29 12:00:00', '2022-01...","['20220129_1200', '20220129_1205', '20220129_1..."
1,124,4959,202201,29,1000,202201,29,1600,165058,996915,NEW YORK,36,2022,January,Blizzard,Z,79,NORTHEAST SUFFOLK,OKX,2022-01-29 10:00:00,EST-5,2022-01-29 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"COOP Observers, trained spotters, NWS employee...",CSV,79,79,NaN,NaN,2022-01-29 15:00:00,2022-01-29 21:00:00,21600.0,MULTIPOLYGON (((-72.46879651799998 40.97624400...,0 days 06:00:00,2022-01-29 15:00:00,2022-01-29 21:00:00,2022-01-29 15:00:00,2022-01-29 21:00:00,"DatetimeIndex(['2022-01-29 15:00:00', '2022-01...","['20220129_1500', '20220129_1505', '20220129_1..."
2,170,6077,202201,29,930,202201,29,1600,165058,996906,NEW YORK,36,2022,January,Blizzard,Z,81,SOUTHEAST SUFFOLK,OKX,2022-01-29 09:30:00,EST-5,2022-01-29 16:00:00,0,0,0,0,0.00K,0.00K,ASOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,Multiple public reports of over 19 inches of s...,CSV,81,81,NaN,NaN,2022-01-29 14:30:00,2022-01-29 21:00:00,23400.0,MULTIPOLYGON (((-72.83286285399998 40.74568176...,0 days 06:30:00,2022-01-29 14:30:00,2022-01-29 21:00:00,2022-01-29 14:30:00,2022-01-29 21:00:00,"DatetimeIndex(['2022-01-29 14:30:00', '2022-01...","['20220129_1430', '20220129_1435', '20220129_1..."


In [59]:
# Run for ONE event (unique by EVENT_ID)

# Will need to streamline this for the entire database of events that we have. But for now, to get code working, keep it simple and focus on one EVENT_ID

### Grab one event to work with
events1 = events_gdf[events_gdf["EVENT_ID"]==996866] # one row of the df, one event ex.

### For blizzards we ran these every 5 min (b/c it was a small number of events). All of these dates are already prepped in that df in the "all_times_format" column

dur = events1["duration"].item()
print(dur)

# read in the df col (which is a list) correctly as a list rather than str
events1["all_times_format"] = events1["all_times_format"].apply(literal_eval)
eventtimes = events1["all_times_format"].item()
print(eventtimes)
print(len(eventtimes)) 

# the eventtimes list, made above, will be used to identify model runs that we have corresponding to those times. Need to search full directory of operational_runs for that (next code cell)

0 days 06:30:00
['20220129_1200', '20220129_1205', '20220129_1210', '20220129_1215', '20220129_1220', '20220129_1225', '20220129_1230', '20220129_1235', '20220129_1240', '20220129_1245', '20220129_1250', '20220129_1255', '20220129_1300', '20220129_1305', '20220129_1310', '20220129_1315', '20220129_1320', '20220129_1325', '20220129_1330', '20220129_1335', '20220129_1340', '20220129_1345', '20220129_1350', '20220129_1355', '20220129_1400', '20220129_1405', '20220129_1410', '20220129_1415', '20220129_1420', '20220129_1425', '20220129_1430', '20220129_1435', '20220129_1440', '20220129_1445', '20220129_1450', '20220129_1455', '20220129_1500', '20220129_1505', '20220129_1510', '20220129_1515', '20220129_1520', '20220129_1525', '20220129_1530', '20220129_1535', '20220129_1540', '20220129_1545', '20220129_1550', '20220129_1555', '20220129_1600', '20220129_1605', '20220129_1610', '20220129_1615', '20220129_1620', '20220129_1625', '20220129_1630', '20220129_1635', '20220129_1640', '20220129_1645

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [109]:
events1.head(4)

,Unnamed: 0.1,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
0,118,4768,202201,29,700,202201,29,1330,165058,996866,NEW YORK,36,2022,January,Blizzard,Z,80,SOUTHWEST SUFFOLK,OKX,2022-01-29 07:00:00,EST-5,2022-01-29 13:30:00,0,0,0,0,0.00K,0.00K,Official NWS Observations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"Islip, NY set a record for its highest calenda...",CSV,80,80,NaN,NaN,2022-01-29 12:00:00,2022-01-29 18:30:00,23400.0,"MULTIPOLYGON (((-73.42420 40.62371, -73.41857 ...",0 days 06:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,"DatetimeIndex(['2022-01-29 12:00:00', '2022-01...","[20220129_1200, 20220129_1205, 20220129_1210, ..."


In [61]:
type(events1)

geopandas.geodataframe.GeoDataFrame

In [108]:
### Find all model run files that correspond to those dates from that event
# Note: will need to repeat this for ODM since that may be beneficial for snow squall/blizzards

alldirs_data_6_ensembling = glob("/home/csutter/DRIVE-clean/operational_runs/*/data_6_ensembling")
print(alldirs_data_6_ensembling[0:3])

allfiles_data_6_ensembling = []
for i in alldirs_data_6_ensembling:
    fs = glob(f"{i}/*/*/*/*/*")
    for f in fs:
        allfiles_data_6_ensembling.append(f)
print(allfiles_data_6_ensembling[0:3])


### Identify files that specifically align with event occurance
# one list will grabs the preduiction file, the other list just tracks the time for the file (not sure if we'll need but might as well)
matched_cnn_file = [] 
matched_cnn_time = []
for model_f in allfiles_data_6_ensembling:
    # parse just the time corresponding to the model pred
    beg = model_f.rfind("/")
    model_time = model_f[beg-13:beg]
    # print(model_time)
    # see if that model_time exists in the list of event times
    # also just have to make sure we already didn't log that file (just in case was accidentally ran twice in operational_run, which can also happen just due to reorg in set__aggregate_allruns, so need to just use one csv pred)
    if ((model_time in eventtimes) & (model_time not in matched_cnn_time)):
        matched_cnn_time.append(model_time) # log the time that matches
        matched_cnn_file.append(model_f) # log the pred file that matches

# print examples and lengts
print("HERE")
print(matched_cnn_time[0:4])
print(len(matched_cnn_time))

print(matched_cnn_file[0:4])
print(len(matched_cnn_file))

# See how this compares to the total duration of times from eventtimes 
print(len(eventtimes))

['/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/data_6_ensembling', '/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling', '/home/csutter/DRIVE-clean/operational_runs/set12_winter2425_NDJFM_000816/data_6_ensembling']
['/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/data_6_ensembling/2025/02/17/20250217_1915/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/data_6_ensembling/2025/02/17/20250217_2345/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set37_lakeeffect/data_6_ensembling/2025/02/17/20250217_1715/finalpreds.csv']
HERE
['20220129_1200', '20220129_1800', '20220129_1300', '20220129_1600']
79
['/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/29/20220129_1200/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/29/20220129_1800/finalpreds.csv', '/home/csutter/DRIVE-clean/operational_ru

In [70]:
# Now, using the relevant model files, for which we know there was XYZ event happening, need to correspond the pred observations of interest to where the event was spatially. This is the spatial join. Since we just simplified it to the EVENT_ID level right now, it's just one region, but note that EPISODE_ID is a weather system, wich will be helpful for visualizations later

# For now, just focus on identifying the cam preds that are IN the weather event

# For each model pred file, spatial join with the event location (geometry) in the events1 df

# Start with one model file for example...

modelfile = pd.read_csv("/home/csutter/DRIVE-clean/operational_runs/set__aggregate_allruns/data_6_ensembling/2022/01/29/20220129_1200/finalpreds.csv")

print(len(modelfile))

# print(list(modelfile.columns))

modelfile_gdf = gpd.GeoDataFrame(
    modelfile, 
    geometry=gpd.points_from_xy(modelfile.Longitude, modelfile.Latitude),
    crs="EPSG:4326" # Start with standard GPS coordinates
)
print(len(modelfile_gdf))

modelfile_gdf = modelfile_gdf.to_crs(events1.crs) # match the CRS to be exactly the system being used in the events dataset
print(len(modelfile_gdf))

# Perform the spatial join
# INNER JOIN for now to limit the result to model preds that were WITHIN the event
# This says: "For every point, find the row in ncei_gdf that contains it"
joined_df = gpd.sjoin(
    modelfile_gdf, 
    events1, 
    how="inner",
    predicate="within" # Check if the point is WITHIN the polygon
)

print(len(joined_df))
display(joined_df.head(4))

2366
2366
2366
59


,Unnamed: 0.3,Unnamed: 0_left,Unnamed: 0.2,Unnamed: 0.1_left,innerPhase_x,img_name,img_orig,site,img_cat,foldnum_x,timeofday,timeofevent,m0_calib_prob_dry,m0_calib_prob_poor_viz,m0_calib_prob_snow,m0_calib_prob_snow_severe,m0_calib_prob_wet,m0_calib_prob,img_orig_hrrr,innerPhase_y,foldnum_y,Latitude,Longitude,yr,mo,day,date,time,ymd,date_and_time_str,date_and_time_dt,date_and_time_dt_round,date_and_time_dt_init,init_hr,init_year,init_month,init_day,init_yyyymmdd,hrrr_file,HRRR_latlon,HRRR_distkm,time.1,valid_time,latitude,longitude,t2m,pt,sh2,d2m,r2,u10,v10,si10,asnow,tp,orog,cape,mslma,dswrf,dlwrf,tcc,gh,dpt,atmosphere,isobaricInhPa,uavg,o_prob_dry,o_prob_poor_viz,o_prob_snow,o_prob_snow_severe,o_prob_wet,o_pred,o_prob,classifier_TF,classifier_01,o_prob_calib,m0_calib_pred,m1_calib_prob_wet,m1_calib_prob_dry,m1_calib_prob_snow,m1_calib_prob_snow_severe,m1_calib_prob_poor_viz,m1_calib_prob,m1_calib_pred,m2_calib_prob_wet,m2_calib_prob_dry,m2_calib_prob_snow,m2_calib_prob_snow_severe,m2_calib_prob_poor_viz,m2_calib_prob,m2_calib_pred,m3_calib_prob_wet,m3_calib_prob_dry,m3_calib_prob_snow,m3_calib_prob_snow_severe,m3_calib_prob_poor_viz,m3_calib_prob,m3_calib_pred,m4_calib_prob_wet,m4_calib_prob_dry,m4_calib_prob_snow,m4_calib_prob_snow_severe,m4_calib_prob_poor_viz,m4_calib_prob,m4_calib_pred,list_5preds,list_5probs,dict_catAsKeys_modelAsValues,dict_catAsKeys_countAsValues,dict_catAsKeys_probsAsValues,dict_mostConfident_singleModel,ensembleAvg_dry,ensembleAvg_snow,ensembleAvg_snow_severe,ensembleAvg_wet,ensembleAvg_poor_viz,ensembleAvg_pred,ensembleAvg_predprob,ensembleMode_pred,ensembleMaxConf_pred,methodalign_1_2,methodalign_1_3,methodalign_2_3,select,decision,select_prob,select_correct,ok_select,num_models_pred_cat,conf_consist,conf_probability,conf_overall,confidence,geometry,index_right,Unnamed: 0.1_right,Unnamed: 0_right,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
54,54,54,44,44,innerTrain,495_Eastbound_at_Exits_65_66_Rest_Area_(Fixed)...,/home/csutter/cron/data/Skyline_1970/20220129/...,Skyline_1970,dry,0,dawn,{Timestamp('2022-01-29 11:35:15.687450'): 'daw...,0.001521,0.231096,0.150306,0.613636,0.003440,0.613636,/home/csutter/cron/data/Skyline_1970/20220129/...,innerTrain,0,40.826437,-72.950239,2022,1,29,20220129,12:00:16,20220129,12:00:16 20220129,2022-01-29 12:00:16,2022-01-29 12:00:00,2022-01-29 10:00:00,10,2022,1,29,20220129,/home/csutter/AI2ES/cleaned/HRRR/2022/01/20220...,"[40.8348754575897, -72.95357036384229]",0.978320,2022-01-29 10:00:00,2022-01-29 12:00:00,40.834875,-72.953570,267.6521,267.7516,0.00230,266.3748,88.8,1.349731,-10.072908,10.137938,0.062958,4.500,29.32589,0.0,100244.0,0.0,274.8,100.0,5310.9250,247.93729,0.0,500.0,10.162935,0.001782,0.270738,0.176090,0.547361,0.004031,snow_severe,0.547361,False,0,0.613636,snow_severe,0.019607,0.007315,0.051367,0.710769,0.210943,0.710769,snow_severe,0.038763,0.010722,0.095321,0.710204,0.144990,0.710204,snow_severe,0.018074,0.006979,0.049752,0.886525,0.038670,0.886525,snow_severe,0.032974,0.004809,0.079659,0.276176,0.606383,0.606383,poor_viz,"['snow_severe', 'snow_severe', 'snow_severe', ...","[0.6136363636363636, 0.7107692307692308, 0.710...","{'snow_severe': ['m0', 'm1', 'm2', 'm3'], 'poo...","{'snow_severe': 4, 'poor_viz': 1}","{'snow_severe': 0.7302836246833208, 

In [102]:
ctsdf = joined_df[["select","img_name"]].groupby(["select"]).count().reset_index()

ctsdf = ctsdf.rename(columns = {"img_name":"count"})

countdict = ctsdf.set_index("select")["count"].to_dict()

display(ctsdf.head(4))
print(countdict)

# other info

id_event = events1["EVENT_ID"].item()
id_ep = events1["EPISODE_ID"].item()
id_type = events1["EVENT_TYPE"].item()

print(id_event,id_ep,id_type)




,select,count
0,poor_viz,5
1,snow_severe,54


{'poor_viz': 5, 'snow_severe': 54}
996866 165058 Blizzard


In [107]:
# Your data
data_to_log = {
    "id_event": id_event,
    "id_ep": id_ep,
    "id_type": id_type,
    "model_counts": countdict # This will be stringified
}

csv_path = "/home/csutter/DRIVE-clean/weather_events/models/stats_events_modelpred/stats.csv"

# Pre-process: Convert the nested dict to a JSON string
data_to_log["model_counts"] = json.dumps(data_to_log["model_counts"])

# Check if file exists to determine if we need a header
file_exists = os.path.isfile(csv_path)

with open(csv_path, mode='a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=data_to_log.keys())
    
    # Write header only if the file is being created for the first time
    if not file_exists:
        writer.writeheader()
        
    writer.writerow(data_to_log)

print(f"Logged event {data_to_log['id_event']} to {csv_path}")

Logged event 996866 to /home/csutter/DRIVE-clean/weather_events/models/stats_events_modelpred/stats.csv


In [ ]:
# What info would we want to save from these simple stats
# 1 - count of cat preds
# 2 - what the weather event was
# 3 - FOR LATER obviously location and stuff but for now keep it simple

In [ ]:
# ANALYSIS 2/6
# Later will add vis and such (TO DO)
# But for now just get some simple stats looping through the different eventsofinterest files and track the stats of proportion of cat preds.
# Put this work in 

# TO DO: I never saved out nonevents in events_ofinterest. Will need to figure out some way to do this cleanly, since it won't have the format of the NCEI events b/c they are NON events. Or maybe just the model run operation_runs directories are enough...

In [68]:
events1

,Unnamed: 0.1,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
0,118,4768,202201,29,700,202201,29,1330,165058,996866,NEW YORK,36,2022,January,Blizzard,Z,80,SOUTHWEST SUFFOLK,OKX,2022-01-29 07:00:00,EST-5,2022-01-29 13:30:00,0,0,0,0,0.00K,0.00K,Official NWS Observations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"Islip, NY set a record for its highest calenda...",CSV,80,80,NaN,NaN,2022-01-29 12:00:00,2022-01-29 18:30:00,23400.0,"MULTIPOLYGON (((-73.42420 40.62371, -73.41857 ...",0 days 06:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,"DatetimeIndex(['2022-01-29 12:00:00', '2022-01...","[20220129_1200, 20220129_1205, 20220129_1210, ..."


In [20]:
#### For identifying / tying eventsofinterest into the geodf (which we need to do to get the geom), use EVENT_ID. 

# Info from gemini (related to that note ^):

# In the NCEI Storm Events Database, Episode ID and Event ID represent a "Parent-Child" relationship. They are designed to group together events that were caused by the same large-scale weather system.

# 1. Episode ID (The Parent)
# An Episode represents a single large-scale weather system or "storm" that affects a broad region (often multiple counties or even multiple states).

# What it groups: It links together all the individual impacts (tornadoes, floods, snow bands) that occurred during that specific storm’s lifespan.

# Uniqueness: The EPISODE_ID is unique to that specific storm system.

# Example: A massive "Winter Storm" moving across New York is assigned one EPISODE_ID.

# 2. Event ID (The Child) -- THIS IS WHAT WE WANT TO JOIN DATABASES BY
# An Event represents a specific type of weather occurring in a specific County or Zone during that Episode.

# What it describes: It is the granular record of what happened at the local level (e.g., "Heavy Snow in Erie County").

# Uniqueness: The EVENT_ID is a globally unique identifier for that specific row in the database. No two rows in the entire NCEI history share the same EVENT_ID.


print(len(d_readin))
print(len(events))
events_withgeom = d_readin[["EVENT_ID", "geometry"]].merge(
    events, 
    on='EVENT_ID', 
    how='right'  # Use 'left' to keep all your geometries
)
print(len(events_withgeom))



7802
14
14


In [ ]:
# Joining spatially (cam lat lon to the event geometries)
# For EACH model run (2400 cam instances)

In [6]:
events.head(4)

,Unnamed: 0.1,Unnamed: 0,BEGIN_YEARMONTH,BEGIN_DAY,BEGIN_TIME,END_YEARMONTH,END_DAY,END_TIME,EPISODE_ID,EVENT_ID,STATE,STATE_FIPS,YEAR,MONTH_NAME,EVENT_TYPE,CZ_TYPE,CZ_FIPS,CZ_NAME,WFO,BEGIN_DATE_TIME,CZ_TIMEZONE,END_DATE_TIME,INJURIES_DIRECT,INJURIES_INDIRECT,DEATHS_DIRECT,DEATHS_INDIRECT,DAMAGE_PROPERTY,DAMAGE_CROPS,SOURCE,MAGNITUDE,MAGNITUDE_TYPE,FLOOD_CAUSE,CATEGORY,TOR_F_SCALE,TOR_LENGTH,TOR_WIDTH,TOR_OTHER_WFO,TOR_OTHER_CZ_STATE,TOR_OTHER_CZ_FIPS,TOR_OTHER_CZ_NAME,BEGIN_RANGE,BEGIN_AZIMUTH,BEGIN_LOCATION,END_RANGE,END_AZIMUTH,END_LOCATION,BEGIN_LAT,BEGIN_LON,END_LAT,END_LON,EPISODE_NARRATIVE,EVENT_NARRATIVE,DATA_SOURCE,CZ_FIPS_FORMAT,ZONE,FIPS,FIPS_FORMAT,BEGIN_UTC,END_UTC,duration_sec,geometry,duration,start_round,end_round,start_buffer,end_buffer,all_times,all_times_format
0,118,4768,202201,29,700,202201,29,1330,165058,996866,NEW YORK,36,2022,January,Blizzard,Z,80,SOUTHWEST SUFFOLK,OKX,2022-01-29 07:00:00,EST-5,2022-01-29 13:30:00,0,0,0,0,0.00K,0.00K,Official NWS Observations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"Islip, NY set a record for its highest calenda...",CSV,80,80,NaN,NaN,2022-01-29 12:00:00,2022-01-29 18:30:00,23400.0,MULTIPOLYGON (((-73.42420196499995 40.62370681...,0 days 06:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,2022-01-29 12:00:00,2022-01-29 18:30:00,"DatetimeIndex(['2022-01-29 12:00:00', '2022-01...","['20220129_1200', '20220129_1205', '20220129_1..."
1,124,4959,202201,29,1000,202201,29,1600,165058,996915,NEW YORK,36,2022,January,Blizzard,Z,79,NORTHEAST SUFFOLK,OKX,2022-01-29 10:00:00,EST-5,2022-01-29 16:00:00,0,0,0,0,0.00K,0.00K,COOP Observer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"COOP Observers, trained spotters, NWS employee...",CSV,79,79,NaN,NaN,2022-01-29 15:00:00,2022-01-29 21:00:00,21600.0,MULTIPOLYGON (((-72.46879651799998 40.97624400...,0 days 06:00:00,2022-01-29 15:00:00,2022-01-29 21:00:00,2022-01-29 15:00:00,2022-01-29 21:00:00,"DatetimeIndex(['2022-01-29 15:00:00', '2022-01...","['20220129_1500', '20220129_1505', '20220129_1..."
2,170,6077,202201,29,930,202201,29,1600,165058,996906,NEW YORK,36,2022,January,Blizzard,Z,81,SOUTHEAST SUFFOLK,OKX,2022-01-29 09:30:00,EST-5,2022-01-29 16:00:00,0,0,0,0,0.00K,0.00K,ASOS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,Multiple public reports of over 19 inches of s...,CSV,81,81,NaN,NaN,2022-01-29 14:30:00,2022-01-29 21:00:00,23400.0,MULTIPOLYGON (((-72.83286285399998 40.74568176...,0 days 06:30:00,2022-01-29 14:30:00,2022-01-29 21:00:00,2022-01-29 14:30:00,2022-01-29 21:00:00,"DatetimeIndex(['2022-01-29 14:30:00', '2022-01...","['20220129_1430', '20220129_1435', '20220129_1..."
3,180,6295,202201,29,700,202201,29,1300,165058,996910,NEW YORK,36,2022,January,Blizzard,Z,78,NORTHWEST SUFFOLK,OKX,2022-01-29 07:00:00,EST-5,2022-01-29 13:00:00,0,0,0,0,0.00K,0.00K,Official NWS Observations,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,A Nor'easter tracked just east of the benchmar...,"Trained spotters, along with surrounding offic...",CSV,78,78,NaN,NaN,2022-01-29 12:00:00,2022-01-29 18:00:00,21600.0,MULTIPOLYGON (((-73.19853210399998 40.86456298...,0 days 06:00:00,2022-01-29 12:00:00,2022-01-29 18:00:00,2022-01-29 12:00:00,2022-01-29 18:00:00,"DatetimeIndex(['2022-01-29 12:00:00', '2022-01...","['20220129_1200', '20220129_1205', '20220129_1..."


In [19]:
events.dtypes

Unnamed: 0.1            int64
Unnamed: 0              int64
BEGIN_YEARMONTH         int64
BEGIN_DAY               int64
BEGIN_TIME              int64
END_YEARMONTH           int64
END_DAY                 int64
END_TIME                int64
EPISODE_ID              int64
EVENT_ID                int64
STATE                  object
STATE_FIPS              int64
YEAR                    int64
MONTH_NAME             object
EVENT_TYPE             object
CZ_TYPE                object
CZ_FIPS                 int64
CZ_NAME                object
WFO                    object
BEGIN_DATE_TIME        object
CZ_TIMEZONE            object
END_DATE_TIME          object
INJURIES_DIRECT         int64
INJURIES_INDIRECT       int64
DEATHS_DIRECT           int64
DEATHS_INDIRECT         int64
DAMAGE_PROPERTY        object
DAMAGE_CROPS           object
SOURCE                 object
MAGNITUDE             float64
MAGNITUDE_TYPE        float64
FLOOD_CAUSE           float64
CATEGORY              float64
TOR_F_SCAL

In [18]:
cams.head(4)

,Unnamed: 0,Latitude,Longitude,ID,Name,DirectionOfTravel,RoadwayName,Url,VideoUrl,Disabled,Blocked
95,95,40.767013,-73.696306,Skyline-1867,I-495 West of New Hyde Park Rd,Eastbound,I-495,https://511ny.org/map/Cctv/1867--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False
96,96,40.770817,-73.687524,Skyline-1868,I-495 at New Hyde Park Rd,Westbound,I-495,https://511ny.org/map/Cctv/1868--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False
97,97,40.774248,-73.670624,Skyline-1869,I-495 at Shelter Rock Rd,Eastbound,I-495,https://511ny.org/map/Cctv/1869--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False
99,99,40.779792,-73.662220,Skyline-1870,I-495 at Searingtown Road,Westbound,I-495,https://511ny.org/map/Cctv/1870--1,https://s52.nysdot.skyvdn.com:443/rtplive/R10_...,False,False


# Old NWS Work

Prepare NWS labeled dataset

['Blizzard Warning', 'Dense Fog Advisory', 'Lake Effect Snow Warning', 'Snow Squall Warning', 'Winter Storm Warning', 'Winter Storm Watch', 'Winter Weather Advisory']

In [2]:
# datetimes and event names (reference code in : /home/csutter/DRIVE-clean/NWS_warnings/notebooks/dataset_analysis.ipynb)
# may take ~40 seconds to read in this data

#### 1. read in data: 
d = gpd.read_file("/home/csutter/DRIVE-clean/NWS_warnings/data/nws_warnings/nws_all_warnings_cleaned.gpkg")

d.head(4)

#### 2. add timedelta duration col back (using duration_sec col)
d["duration"] = pd.to_timedelta(d["duration_sec"], unit="s")

#### 3. Subset to squalls (or whatever event type)
d_event_subset = d[d["name"]=='Snow Squall Warning'] 


#### 4. Add in range of 5-min intervals in entire duration of warning

d_event_subset["start_round"] = d_event_subset["issued"].dt.round("5min") # will want these in a df for reference of the rounded start time
d_event_subset["end_round"] = d_event_subset["expired"].dt.round("5min") # will want these in a df for reference of the rounded ende time

d_event_subset["start_buffer"] = d_event_subset["start_round"] - pd.Timedelta(minutes=15)
d_event_subset["end_buffer"] = d_event_subset["end_round"] + pd.Timedelta(minutes=15)

d_event_subset["all_times"] = d_event_subset.apply(
    lambda row: pd.date_range(start=row["start_buffer"], end=row["end_buffer"], freq="5T"),
    axis=1
)

d_event_subset["all_times_format"] = d_event_subset["all_times"].apply(
    lambda elem: [t.strftime("%Y%m%d_%H%M") for t in elem]
)

d_event_subset.head(4)


#### 5. make a list of all the datetimes we want to run
# nested loop, bc each row (squall instance) has a range of datetimes to run

nws_datetimes = []

for i in d_event_subset["all_times_format"]:
    for j in i:
        nws_datetimes.append(j)

#### 6. CRITICAL - subset to unique datetimes! Events in different regions will have overlapping times of warning. Since we run events statewide, only need the datetime once (i.e., it's not datetime|location)

nws_datetimes = np.unique(nws_datetimes)

print(len(nws_datetimes))

/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value ins

1089


/usr/local/lib/python3.8/dist-packages/geopandas/geodataframe.py:1443: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


Grab list of datetimes already ran in inference


In [ ]:
# Grab list of all aggregated datetimes (I have reference code for this, /home/csutter/DRIVE-clean/operational_analysis/notebooks/summarize_dates_ran.ipynb)

In [11]:
inf_sets_ran = ["/home/csutter/DRIVE-clean/operational_runs/set0_test",
"/home/csutter/DRIVE-clean/operational_runs/set1_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set2_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set3_20250924",
"/home/csutter/DRIVE-clean/operational_runs/set4_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set5_iceWTA_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set6_probSRsample_20250925",
"/home/csutter/DRIVE-clean/operational_runs/set7_probSRsample_20250926",
"/home/csutter/DRIVE-clean/operational_runs/set9_winter22_JFMOnly_000816",
"/home/csutter/DRIVE-clean/operational_runs/set10_winter2223_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set11_winer2324_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set12_winter2425_NDJFM_000816",
"/home/csutter/DRIVE-clean/operational_runs/set13_winter22_JFMOnly_041220",
"/home/csutter/DRIVE-clean/operational_runs/set14_winter2223_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set15_winter2324_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set16_winter2425_NDJFM_041220",
"/home/csutter/DRIVE-clean/operational_runs/set17_summer_examples",
"/home/csutter/DRIVE-clean/operational_runs/set18_summer_examples_2",
"/home/csutter/DRIVE-clean/operational_runs/set19_summer_examples_3",
"/home/csutter/DRIVE-clean/operational_runs/set20_summer_examples_4",
"/home/csutter/DRIVE-clean/operational_runs/set21_summer_examples_5",
"/home/csutter/DRIVE-clean/operational_runs/set22_summer_examples_6",
"/home/csutter/DRIVE-clean/operational_runs/set23_squall1",
"/home/csutter/DRIVE-clean/operational_runs/set24_squall2",
"/home/csutter/DRIVE-clean/operational_runs/set25_squall3",
"/home/csutter/DRIVE-clean/operational_runs/set26_blizzard1",
"/home/csutter/DRIVE-clean/operational_runs/set27_blizzard2",
"/home/csutter/DRIVE-clean/operational_runs/set28_blizzard3",
"/home/csutter/DRIVE-clean/operational_runs/set29_blizzard4",
"/home/csutter/DRIVE-clean/operational_runs/set30_blizzard5",]

dirs_w_preds = [f"{i}/data_6_ensembling/*/*/*/*/*" for i in inf_sets_ran]

# print(dirs_w_preds)

pred_datetimes = []
pred_path = []
for dr in dirs_w_preds:
    # print(dr) # just for checking counts per dir
    # dr_files = [] # just for checking counts per dir
    listpredfiles = glob(dr)
    for fl in listpredfiles:
        # print(fl)
        i = fl.rfind("/")
        filedate = fl[i-13:i]
        pred_datetimes.append(filedate)
        # dr_files.append(filedate) # just for checking counts per dir

        ### Must also grab the tracker path which contains the predictions and the image path (should have all the data in it we need)
        pred_path.append(fl)
    # print(len(dr_files)) # just for checking counts per dir


print(len(pred_datetimes))
print(len(pred_path))


4924
4924


In [5]:
pred_datetimes[0:5]

['20250210_1000',
 '20250204_1000',
 '20250228_1000',
 '20250227_1000',
 '20250211_1000']

In [12]:
pred_path[0:5]

['/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/10/20250210_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/04/20250204_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/28/20250228_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/27/20250227_1000/finalpreds.csv',
 '/home/csutter/DRIVE-clean/operational_runs/set0_test/data_6_ensembling/2025/02/11/20250211_1000/finalpreds.csv']

Colocate nws event with existant inference obersvations (i.e. the already ran model runs, found above)
- Something to think about - How to deal with onset? Like a winter storm warning can encompass days... so maybe it's just applicable to snow squalls/other short lived events. 
- Application wise, do I need to use the entire EVENT or can I grab random 5-min timestamps within the event? And what if the squall was in the beginning but the warning just prolongs for public safety?

In [8]:
# Identify dates in both the nws_events and model_obs lists

nwsevent_with_preds = [d for d in pred_datetimes if d in nws_datetimes]
nonnwsevent_with_preds = [d for d in pred_datetimes if d not in nws_datetimes]

print(len(nwsevent_with_preds))
print(len(nonnwsevent_with_preds))

722
4195


Colocate cam data (lat / lon) with NWS event polygon

In [10]:
# need to use the raw dataframes (not just list of dates) for that
# NWS events df: d_event_subset, note that this contains all of the events which are unique by geometry|event|timeframe
# Model preds df: VARIOUS of them, need to see list of them; the list of the pred csvs are in the list called pred_path

d_event_subset.head(4)
print(type(d_event_subset))
print(d_event_subset.dtypes)

<class 'geopandas.geodataframe.GeoDataFrame'>
Unnamed: 0                    int64
vtec_year                     int64
iso_issued                   object
issued               datetime64[ns]
iso_expired                  object
expired              datetime64[ns]
eventid                       int64
phenomena                    object
significance                 object
hvtec_nwsli                  object
wfo                          object
ugc                          object
product_id                   object
name                         object
ph_name                      object
sig_name                     object
url                          object
location_type                object
ugc_gis                      object
loc_desc                     object
duration_sec                float64
geometry                   geometry
duration            timedelta64[ns]
start_round          datetime64[ns]
end_round            datetime64[ns]
start_buffer         datetime64[ns]
end_buffer        

Sample so that events and non-events are represented
- If we don't have all, can just start w snow squalls as a "proof of concept" that this idea may work...

Gather labeled dataset for model training
- cols for model inputs: img-only pred (5-cat), final pred (5-cat), nonobs pred (2-cat), raw weather data (6 vars) (rational for weather data is that, even tho this data is embedded in the final pred, the model still has errors so maybe adding in the weather data - which esp makes sense given it's nws weather events we're predicting - will help w extraneous/fine tuning. O/w, why would we expect the img-only pred or final pred to align exactly w nws events? it's different data.)
- cols for model outputs (nws event label): one hot encode or whatever

Preprocess data
- normalize, etc. 

Train model
- For now, just proof of concept w a random forest (something easy)